[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/agents/openai/openai_agents_parallelization.ipynb)


# <center>OpenAI 智能体模式：并行执行</center>

本入门指南介绍如何使用 `openai-agents` 库构建支持并行 LLM 与工具调用的智能体。

本例构建一个股票研究智能体。研究 5 只股票时，可以让智能体并行执行多次工具调用。

### 安装库

In [ ]:
# 安装 OpenAI 基础库
!pip install -q openai openai-agents yfinance

# 安装 OpenInference/OpenTelemetry 追踪的可选库
!pip install -q arize-phoenix-otel openinference-instrumentation-openai-agents openinference-instrumentation-openai

### 设置密钥
将 OpenAI API 密钥添加到环境变量 `OPENAI_API_KEY`。

从 [app.phoenix.arize.com](https://app.phoenix.arize.com) 的设置页面复制 Phoenix `API_KEY`。

In [ ]:
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("🔑 请输入 OpenAI API 密钥： ")

if "PHOENIX_API_KEY" not in os.environ:
    os.environ["PHOENIX_API_KEY"] = getpass("🔑 请输入 Phoenix API 密钥： ")

if "PHOENIX_COLLECTOR_ENDPOINT" not in os.environ:
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = getpass("🔑 请输入 Phoenix 收集器端点： ")

### 设置追踪

In [ ]:
from phoenix.otel import register

tracer_provider = register(
    project_name="openai-agents",
    auto_instrument=True,
)

## 创建智能体

In [ ]:
from pprint import pprint

import yfinance as yf


def get_stock_data(ticker_symbol):
    stock = yf.Ticker(ticker_symbol)
    data = stock.info
    return {
        "symbol": data.get("symbol"),
        "current_price": data.get("currentPrice"),
        "market_cap": data.get("marketCap"),
        "sector": data.get("sector"),
        "industry": data.get("industry"),
        "description": data.get("longBusinessSummary"),
        "trailing_pe": data.get("trailingPE"),
        "forward_pe": data.get("forwardPE"),
        "dividend_yield": data.get("dividendYield"),
        "beta": data.get("beta"),
        "fifty_two_week_high": data.get("fiftyTwoWeekHigh"),
        "fifty_two_week_low": data.get("fiftyTwoWeekLow"),
        "fifty_day_moving_average": data.get("fiftyDayAverage"),
        "two_hundred_day_moving_average": data.get("twoHundredDayAverage"),
        "recommendation_key": data.get("recommendationKey"),
        "revenue_growth": data.get("revenueGrowth"),
        "earnings_growth": data.get("earningsGrowth"),
        "profit_margins": data.get("profitMargins"),
    }


pprint(get_stock_data("AAPL"))

In [ ]:
from pprint import pprint
from textwrap import dedent

from agents import Agent, Runner, TResponseInputItem, WebSearchTool, function_tool
from agents.model_settings import ModelSettings


@function_tool
def get_stock_data_tool(ticker_symbol: str) -> dict:
    """
    获取指定股票代码的股票数据。
    Args:
        ticker_symbol: 要获取数据的股票代码。
    Returns:
        包含价格、市值等股票数据的字典。
    """
    return get_stock_data(ticker_symbol)


research_agent = Agent(
    name="FinancialSearchAgent",
    instructions=dedent(
        """你是专注金融主题的研究助手。给定股票代码后，使用网络搜索获取最新背景，生成不超过 50 个词的简短摘要，重点关注对金融分析师有用的关键数字、事件或引述。"""
    ),
    model="gpt-4.1",
    tools=[WebSearchTool(), get_stock_data_tool],
    model_settings=ModelSettings(tool_choice="required", parallel_tool_calls=True),
)

user_input = input("请输入要研究的股票代码： ")
input_items: list[TResponseInputItem] = [{"content": user_input, "role": "user"}]

orchestrator = await Runner.run(research_agent, input_items)
orchestrator_output = orchestrator.final_output
pprint(orchestrator_output)